# CREAR UNA RED NEURONAL DE EMBEDDINGS CON PYTORCH

# PASO 1 - IMPORTAMOS LIBRERIAS

In [1]:
import torch
import torch.nn as nn

# PASO 2 - CREAMOS DATASET DE EJEMPLO

In [2]:
# Dataset de ejemplo
sentences = [
    ["I", "love", "deep", "learning"],
    ["I", "love", "machine", "learning"],
    ["deep", "learning", "is", "fun"]
]

In [3]:
# Crear un vocabulario simple
vocab = {"<PAD>": 0, "I": 1, "love": 2, "deep": 3, "learning": 4, "machine": 5, "is": 6, "fun": 7}

In [4]:
# Convertir las palabras a índices en el vocabulario
indexed_sentences = [[vocab[word] for word in sentence] for sentence in sentences]

In [5]:
print("Indexed sentences:", indexed_sentences)

Indexed sentences: [[1, 2, 3, 4], [1, 2, 5, 4], [3, 4, 6, 7]]


# CREAMOS UN CAPA PARA LOS EMBEDDINGS

In [6]:
# Parámetros de la capa de embeddings
embedding_dim = 8
vocab_size = len(vocab)

In [7]:
# Crear la capa de embeddings
embedding_layer = nn.Embedding(vocab_size, embedding_dim)
# Ver los parámetros de la capa de embeddings
print("Embeddings matrix shape:", embedding_layer.weight.shape)

Embeddings matrix shape: torch.Size([8, 8])


In [8]:
# Convertir las frases indexadas en tensores de PyTorch
input_tensor = torch.tensor(indexed_sentences)

In [9]:
input_tensor

tensor([[1, 2, 3, 4],
        [1, 2, 5, 4],
        [3, 4, 6, 7]])

In [10]:
# Obtener las representaciones de las palabras
embedded_sentences = embedding_layer(input_tensor)

In [11]:
embedded_sentences

tensor([[[ 0.7743,  1.3954,  0.3529,  0.9307,  0.6695, -0.2049,  0.7394,
          -1.0767],
         [ 1.1894, -0.5060, -0.0842, -0.0125,  1.4162, -0.1866,  1.3899,
           1.2810],
         [-0.9966,  0.5088,  0.1551, -0.2081,  1.2103,  1.5058, -0.1493,
           0.5035],
         [ 0.3006, -2.0523,  0.5394,  0.3338, -1.0558, -0.4450, -0.2738,
          -0.7785]],

        [[ 0.7743,  1.3954,  0.3529,  0.9307,  0.6695, -0.2049,  0.7394,
          -1.0767],
         [ 1.1894, -0.5060, -0.0842, -0.0125,  1.4162, -0.1866,  1.3899,
           1.2810],
         [-1.6099,  1.0741, -0.0496,  0.2555, -1.2973,  0.9995, -0.4991,
           1.7835],
         [ 0.3006, -2.0523,  0.5394,  0.3338, -1.0558, -0.4450, -0.2738,
          -0.7785]],

        [[-0.9966,  0.5088,  0.1551, -0.2081,  1.2103,  1.5058, -0.1493,
           0.5035],
         [ 0.3006, -2.0523,  0.5394,  0.3338, -1.0558, -0.4450, -0.2738,
          -0.7785],
         [ 0.5960,  1.6387,  0.2303,  0.6645,  0.2134, -2.1966,  2

# CREAMOS AL RED NEURONAL DE EMBEDDING


In [12]:
class SimpleModel(nn.Module):
    def __init__(self,vocab_size,embedding_dim):
      super(SimpleModel,self).__init__()
      self.embeddings = nn.Embedding(vocab_size,embedding_dim)
      self.fc = nn.Linear(embedding_dim,1) # clasificador básico

    def forward(self,x):
      x = self.embeddings(x)
      x = x.mean(dim=1)
      return self.fc(x)

In [13]:
model = SimpleModel(vocab_size,embedding_dim)
output = model(input_tensor)
print("Output model:", output)
print("Output shape :",output.shape)

Output model: tensor([[ 0.2265],
        [ 0.4127],
        [-0.3355]], grad_fn=<AddmmBackward0>)
Output shape : torch.Size([3, 1])


In [14]:
import torch.optim as optim

# Definir una función de pérdida y un optimizador
criterion = nn.MSELoss() # Error cuadrático medio para regresión
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Crear etiquetas dummy para el entrenamiento (en un escenario real, estas serían tus etiquetas verdaderas)
target_labels = torch.randn(3, 1) # 3 ejemplos, 1 etiqueta por ejemplo

print("\n--- Entrenamiento del Modelo ---")
num_epochs = 10
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(input_tensor)
    loss = criterion(outputs, target_labels)

    # Backward y optimización
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("\n--- Predicciones después del Entrenamiento ---")
# Hacer una predicción con el modelo entrenado
with torch.no_grad(): # Desactiva el cálculo de gradientes para la inferencia
    predictions = model(input_tensor)
print("Predicciones (después de entrenamiento):\n", predictions)
print("Shape de las predicciones:", predictions.shape)


--- Entrenamiento del Modelo ---
Epoch [1/10], Loss: 0.7859
Epoch [2/10], Loss: 0.7300
Epoch [3/10], Loss: 0.6819
Epoch [4/10], Loss: 0.6407
Epoch [5/10], Loss: 0.6059
Epoch [6/10], Loss: 0.5767
Epoch [7/10], Loss: 0.5526
Epoch [8/10], Loss: 0.5329
Epoch [9/10], Loss: 0.5169
Epoch [10/10], Loss: 0.5038

--- Predicciones después del Entrenamiento ---
Predicciones (después de entrenamiento):
 tensor([[-0.2627],
        [-0.1966],
        [-0.6129]])
Shape de las predicciones: torch.Size([3, 1])
